# Проверка и объединение Реестра ППЗ (2023–2025)

Три файла `.xls` по годам объединяются в единый датасет.
Проводятся проверки качества данных и сопоставление с CRM.

## 0. Импорт и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import warnings

# Подавляем предупреждения pandas/openpyxl, чтобы вывод был чистым
warnings.filterwarnings("ignore")

# Расширяем отображение таблиц в Jupyter: больше колонок, строк,
# полные тексты без обрезки — удобно при первом осмотре данных
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

# ── Пути к файлам ──
# DATA_DIR — папка на сервере, где лежат исходные файлы.
# ⚠ Замените пути PPZ_FILES на реальные имена файлов после копирования.
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"

# Словарь: год → путь к соответствующему XLS-файлу Реестра ППЗ
PPZ_FILES = {
    2023: f"{DATA_DIR}/ppz_2023.xls",
    2024: f"{DATA_DIR}/ppz_2024.xls",
    2025: f"{DATA_DIR}/ppz_2025.xls",
}

# CRM-выгрузка — нужна для сопоставления ИНН в секции 4
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

# Путь для сохранения объединённого файла ППЗ
OUT_FILE = f"{DATA_DIR}/ppz_registry_combined.csv"

# Допустимый диапазон дат: все записи должны попадать в этот период
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

# ── Параметры чтения XLS ──
# SKIP_ROWS — сколько строк пропустить в начале файла (шапка отчёта).
# Если шапка — 1 строка с названием отчёта, ставим 1.
# Если 2 строки (название + пустая), ставим 2 и т.д.
# ⚠ Подберите значение по результату: строка с "Филиал", "Клиентский менеджер"
#   должна стать заголовком (header), а не данными.
SKIP_ROWS = 3


def normalize_inn(s):
    """Приводит ИНН к единому строковому формату.

    Проблема: при чтении из Excel числовой ИНН (например, 7701234567)
    может прийти как '7701234567.0' или потерять ведущие нули.
    Функция:
      1. Убирает суффикс '.0' (артефакт Excel → float → string)
      2. Убирает ведущие нули (чтобы затем дополнить до нужной длины)
      3. Дополняет нулями слева: до 10 знаков (юрлицо) или 12 (ИП)
    """
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("Конфигурация задана.")
for y, p in PPZ_FILES.items():
    print(f"  {y}: {p}")

## 1. Загрузка и осмотр структуры

Читаем каждый файл отдельно, выводим shape, колонки и dtypes.
Сравниваем набор колонок между годами.

In [ ]:
# Словарь frames: год → DataFrame с данными этого года.
# Каждый файл читается отдельно, чтобы сначала сравнить структуру,
# а потом уже объединять.
frames = {}

for year, path in PPZ_FILES.items():
    # dtype=str — читаем ВСЁ как строки, чтобы не потерять ведущие нули ИНН,
    # не перепутать коды с числами и не сломать даты при автоконвертации.
    # skiprows=SKIP_ROWS — пропускаем шапку отчёта (название, логотипы и т.п.),
    # чтобы pandas увидел строку с реальными именами колонок как header.
    tmp = pd.read_excel(path, dtype=str, skiprows=SKIP_ROWS)

    # Приводим названия колонок к строкам (Excel может вернуть float/int)
    # и убираем случайные пробелы
    tmp.columns = [str(c).strip() for c in tmp.columns]

    # Удаляем пустые колонки «Unnamed: N» или «nan» — артефакт Excel,
    # когда в файле есть пустые столбцы правее данных
    unnamed = [c for c in tmp.columns if c.startswith("Unnamed") or c == "nan"]
    if unnamed:
        print(f"  {year}: удалены пустые колонки: {unnamed}")
        tmp = tmp.drop(columns=unnamed)

    # Удаляем строку-нумерацию из Excel (первая строка после заголовка,
    # содержащая числа 0, 1, 2, 3... — порядковые номера столбцов).
    # Определяем её по тому, что большинство значений — целые числа от 0 до 30.
    first_row = tmp.iloc[0]
    numeric_vals = pd.to_numeric(first_row, errors="coerce").dropna()
    if len(numeric_vals) > len(first_row) * 0.5 and numeric_vals.max() < 50:
        tmp = tmp.iloc[1:].reset_index(drop=True)
        print(f"  {year}: удалена строка-нумерация столбцов из Excel")

    # Удаляем полностью пустые строки (могут быть из-за шапки или разделителей)
    before = len(tmp)
    tmp = tmp.dropna(how="all")
    dropped = before - len(tmp)
    if dropped > 0:
        print(f"  {year}: удалено {dropped} полностью пустых строк")

    frames[year] = tmp

    # Выводим базовую информацию: размер, названия колонок, типы, первые строки
    print(f"\n{'='*60}")
    print(f"  {year}: {len(tmp):,} строк × {tmp.shape[1]} колонок")
    print(f"{'='*60}")
    print(f"Колонки: {list(tmp.columns)}")
    print(f"\ndtypes:\n{tmp.dtypes.to_string()}")
    print(f"\nПервые 3 строки:")
    display(tmp.head(3))

In [ ]:
# ── Проверка совпадения колонок между годами ──
# Если в одном из файлов добавили/убрали колонку, pd.concat заполнит
# недостающие значения NaN-ами — важно знать об этом заранее.

years = sorted(frames.keys())
# Для каждого года получаем множество (set) названий колонок
col_sets = {y: set(frames[y].columns) for y in years}

# Объединение всех колонок (для информации) и пересечение (общие для всех)
all_cols = set()
for s in col_sets.values():
    all_cols |= s
common = set.intersection(*col_sets.values())

print(f"Общие колонки ({len(common)}): {sorted(common)}")

# Проверяем, есть ли колонки, которые существуют только в одном из файлов
has_diff = False
for y in years:
    only_here = col_sets[y] - common
    if only_here:
        has_diff = True
        print(f"\nТолько в {y}: {sorted(only_here)}")

if not has_diff:
    print("\nРасхождений нет — колонки совпадают во всех файлах.")

## 2. Объединение в единый датасет

In [ ]:
# Добавляем в каждый DataFrame колонку source_year — для трассировки:
# чтобы после объединения можно было понять, из какого файла пришла строка
parts = []
for year, tmp in frames.items():
    tmp = tmp.copy()
    tmp["source_year"] = year
    parts.append(tmp)

# pd.concat — вертикальное объединение (стэкинг) трёх таблиц в одну.
# ignore_index=True — сбрасываем индексы, чтобы получить сквозную нумерацию 0…N
df = pd.concat(parts, ignore_index=True)
print(f"Объединённый датасет: {len(df):,} строк × {df.shape[1]} колонок")
print(f"\nСтрок по годам:")
print(df["source_year"].value_counts().sort_index().to_string())

In [ ]:
# Сохраняем объединённый датасет в CSV:
#   sep=";"        — разделитель «точка с запятой» (стандарт для русскоязычных данных,
#                    т.к. в числах используется запятая как десятичный разделитель)
#   encoding="utf-8-sig" — UTF-8 с BOM-меткой, чтобы Excel корректно открывал
#                          файл с кириллицей без ручного выбора кодировки
#   index=False    — не записываем служебный индекс pandas в файл
df.to_csv(OUT_FILE, index=False, sep=";", encoding="utf-8-sig")
print(f"Сохранено: {OUT_FILE}")

## 3. Проверки качества данных

### 3.1. Период данных (2023–2025)

Ищем все колонки, содержащие даты, и проверяем их диапазон.

In [ ]:
# ── Автоматическое определение колонок с датами ──
# Мы не знаем заранее, как называются колонки с датами в ППЗ.
# Поэтому берём первые 200 непустых значений каждой колонки и пробуем
# распарсить их как дату. Если >70% значений успешно парсятся — считаем
# колонку «датной».
date_cols = []
for col in df.columns:
    if col == "source_year":
        continue
    sample = df[col].dropna().head(200)
    if len(sample) == 0:
        continue
    # dayfirst=True — формат ДД.ММ.ГГГГ (российский стандарт)
    # errors="coerce" — непарсящиеся значения становятся NaT (а не ошибкой)
    parsed = pd.to_datetime(sample, dayfirst=True, errors="coerce")
    ratio = parsed.notna().mean()
    if ratio > 0.7:
        date_cols.append(col)

print(f"Обнаружено колонок с датами: {len(date_cols)}")
print(f"Колонки: {date_cols}")

# Для каждой «датной» колонки проверяем:
#   - сколько значений успешно распарсились
#   - сколько не распарсились (NaT, но при этом исходное значение не пустое)
#   - минимальная и максимальная дата
#   - сколько записей выходит за пределы допустимого периода 2023–2025
for col in date_cols:
    dt = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
    valid = dt.dropna()
    out_of_range = valid[(valid < DATE_FROM) | (valid > DATE_TO)]
    print(f"\n--- {col} ---")
    print(f"  Всего значений: {len(valid):,}")
    print(f"  Не распарсилось (NaT): {dt.isna().sum() - df[col].isna().sum():,}")
    print(f"  Мин: {valid.min():%d.%m.%Y}  |  Макс: {valid.max():%d.%m.%Y}")
    print(f"  За пределами 2023–2025: {len(out_of_range):,}")
    if len(out_of_range) > 0 and len(out_of_range) <= 20:
        print(f"  Примеры: {out_of_range.head(10).dt.strftime('%d.%m.%Y').tolist()}")

### 3.2. Пропуски (nulls)

In [ ]:
# ── Анализ пропусков (null / NaN) по каждой колонке ──
# Цель: понять, какие колонки хорошо заполнены, а какие почти пустые.
# Колонки с >50% пропусков — потенциально бесполезные или требуют уточнения
# у источника данных.

nulls = df.isnull().sum()          # количество NaN в каждой колонке
total = len(df)                     # общее число строк

# Собираем отчёт: колонка → кол-во пропусков → процент пропусков
null_report = pd.DataFrame({
    "Колонка": nulls.index,
    "Пропуски": nulls.values,
    "% пропусков": (nulls.values / total * 100).round(1),
}).sort_values("% пропусков", ascending=False).reset_index(drop=True)

print(f"Всего строк: {total:,}\n")

# Выводим таблицу с цветовой шкалой: чем больше пропусков, тем краснее ячейка
display(
    null_report.style
    .hide(axis="index")
    .background_gradient(subset=["% пропусков"], cmap="YlOrRd")
)

# Отдельно выделяем «проблемные» колонки с более чем 50% пустых значений
bad_cols = null_report[null_report["% пропусков"] > 50]
if len(bad_cols) > 0:
    print(f"\nКолонки с >50% пропусков ({len(bad_cols)}):")
    for _, row in bad_cols.iterrows():
        print(f"  • {row['Колонка']} — {row['% пропусков']}%")
else:
    print("\nКолонок с >50% пропусков нет.")

### 3.3. Дубликаты

In [ ]:
# ── Поиск дубликатов ──
# Проверяем два сценария:
#   1. Полные дубликаты — строки, идентичные по ВСЕМ колонкам (кроме source_year,
#      т.к. она добавлена нами). Такие строки — точные копии.
#   2. Кросс-годовые дубликаты — одна и та же запись попала в файлы разных лет
#      (например, запись за дек. 2023 есть и в файле 2023, и в файле 2024).

# Исключаем source_year из сравнения — она всегда различается между годами
cols_no_year = [c for c in df.columns if c != "source_year"]

# duplicated() возвращает True для каждой повторяющейся строки (кроме первого вхождения)
full_dupes = df.duplicated(subset=cols_no_year).sum()
print(f"Полных дубликатов (без source_year): {full_dupes:,} из {len(df):,} ({full_dupes/len(df)*100:.1f}%)")

# keep=False — помечаем ВСЕ строки дубликата (включая первое вхождение),
# чтобы увидеть, из каких годов они пришли
cross_year_dupes = df[df.duplicated(subset=cols_no_year, keep=False)]
if len(cross_year_dupes) > 0:
    cross = cross_year_dupes.groupby("source_year").size()
    print(f"\nДубликаты по годам (строки, участвующие в дублировании):")
    print(cross.to_string())
    print(f"\nПримеры дубликатов (первые 5):")
    display(cross_year_dupes.head(5))

### 3.4. Формат ИНН

In [ ]:
# ── Поиск и валидация колонки с ИНН ──
# Ищем колонку по ключевым словам в названии (на русском и английском).
# Если не находим — выводим все колонки для ручного выбора.
inn_candidates = [c for c in df.columns if "инн" in c.lower() or "inn" in c.lower()]
print(f"Колонки-кандидаты для ИНН: {inn_candidates}")

if len(inn_candidates) == 0:
    print("\n⚠ Колонка с ИНН не найдена автоматически.")
    print("Все колонки:", list(df.columns))
    INN_COL = None
else:
    INN_COL = inn_candidates[0]
    print(f"\nИспользуется: '{INN_COL}'")

if INN_COL:
    raw = df[INN_COL].copy()

    # 1. Пропуски — строки без ИНН вообще
    null_inn = raw.isna().sum()
    print(f"\nПропусков ИНН: {null_inn:,}")

    # 2. Нормализация — приводим к единому формату (10 или 12 цифр)
    normed = raw.apply(normalize_inn)

    # 3. Проверка длины — ИНН юрлица = 10 знаков, ИП = 12 знаков.
    #    Любая другая длина — ошибка в данных.
    lengths = normed.dropna().str.len().value_counts().sort_index()
    print(f"\nРаспределение длин ИНН (после нормализации):")
    print(lengths.to_string())

    bad_len = normed.dropna()[~normed.dropna().str.len().isin([10, 12])]
    if len(bad_len) > 0:
        print(f"\n⚠ ИНН с некорректной длиной (не 10/12): {len(bad_len):,}")
        print(f"  Примеры: {bad_len.head(10).tolist()}")

    # 4. Проверка на нецифровые символы — ИНН должен содержать только цифры
    non_digit = normed.dropna()[~normed.dropna().str.match(r'^\d+$')]
    if len(non_digit) > 0:
        print(f"\n⚠ ИНН с нецифровыми символами: {len(non_digit):,}")
        print(f"  Примеры: {non_digit.head(10).tolist()}")
    else:
        print("\nВсе ИНН содержат только цифры.")

    # 5. Количество уникальных компаний в датасете
    n_unique = normed.dropna().nunique()
    print(f"\nУникальных ИНН: {n_unique:,}")

### 3.5. Формат дат

In [ ]:
# ── Детальная проверка формата дат ──
# Для каждой «датной» колонки считаем:
#   - сколько значений заполнено
#   - сколько НЕ распарсились (значение есть, но формат неизвестен) —
#     это разница между NaT после парсинга и исходными NaN
#   - выводим примеры проблемных значений для ручного анализа

if len(date_cols) == 0:
    print("Колонки с датами не обнаружены (см. секцию 3.1).")
else:
    for col in date_cols:
        dt = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
        # unparsed — значения, которые НЕ пустые в исходных данных,
        # но стали NaT после парсинга (т.е. нераспознанный формат даты)
        unparsed = dt.isna().sum() - df[col].isna().sum()
        print(f"\n--- {col} ---")
        print(f"  Заполнено: {df[col].notna().sum():,}")
        print(f"  Не распарсилось: {unparsed:,}")
        if unparsed > 0:
            bad_mask = dt.isna() & df[col].notna()
            print(f"  Примеры проблемных значений: {df.loc[bad_mask, col].head(10).tolist()}")

    # Если есть ≥2 датных колонки — проверяем логику:
    # например, «дата начала» не должна быть позже «даты окончания».
    # Мы не знаем заранее, какая колонка — начало, а какая — конец,
    # поэтому проверяем все пары и выводим количество нарушений.
    if len(date_cols) >= 2:
        print(f"\n\n--- Проверка логики дат (попарно) ---")
        for i in range(len(date_cols)):
            for j in range(i + 1, len(date_cols)):
                c1, c2 = date_cols[i], date_cols[j]
                d1 = pd.to_datetime(df[c1], dayfirst=True, errors="coerce")
                d2 = pd.to_datetime(df[c2], dayfirst=True, errors="coerce")
                # Сравниваем только строки, где обе даты заполнены
                both = d1.notna() & d2.notna()
                if both.sum() == 0:
                    continue
                wrong = (d1[both] > d2[both]).sum()
                print(f"\n  {c1} > {c2}: {wrong:,} из {both.sum():,} ({wrong/both.sum()*100:.1f}%)")

### 3.6. Категориальные поля

In [ ]:
# ── Анализ категориальных (справочных) полей ──
# Категориальные — это колонки с небольшим числом уникальных значений
# (от 2 до MAX_UNIQUE). Например: тип фактора, статус, сегмент.
# Для них выводим частоту каждого значения и проверяем на «грязь»:
# лишние пробелы, опечатки, дублирование из-за регистра.

MAX_UNIQUE = 30  # порог: если уникальных значений ≤ 30 — считаем категориальной

cat_cols = []
for col in df.columns:
    # Пропускаем служебные, датовые и ИНН-колонки
    if col in ("source_year",) or col in date_cols:
        continue
    if col == INN_COL:
        continue
    n_uniq = df[col].nunique()
    if 2 <= n_uniq <= MAX_UNIQUE:
        cat_cols.append(col)

print(f"Категориальных колонок (2–{MAX_UNIQUE} уник. значений): {len(cat_cols)}")

for col in cat_cols:
    # value_counts — частота каждого значения; dropna=False показывает и NaN
    print(f"\n--- {col} ({df[col].nunique()} уник.) ---")
    vc = df[col].value_counts(dropna=False).head(15)
    print(vc.to_string())

    # Проверка на лишние пробелы: "Да" и "Да " — это разные значения в pandas,
    # но по смыслу одинаковые. Такие артефакты ломают группировки.
    stripped = df[col].dropna().str.strip()
    if not stripped.equals(df[col].dropna()):
        diff = (stripped != df[col].dropna()).sum()
        print(f"  ⚠ Есть лишние пробелы: {diff:,} записей")

### 3.7. Числовые поля

In [ ]:
# ── Анализ числовых полей ──
# Числовые — колонки, которые не попали в категориальные и не являются датами/ИНН,
# но при этом >80% значений успешно конвертируются в числа.
# Выводим describe() (мин, макс, среднее, квартили) и проверяем
# на отрицательные значения (если это суммы/количества — они не должны быть < 0).

num_cols = []
for col in df.columns:
    if col in ("source_year",) or col in date_cols or col == INN_COL:
        continue
    if col in cat_cols:
        continue
    sample = df[col].dropna().head(200)
    if len(sample) == 0:
        continue
    numeric = pd.to_numeric(sample, errors="coerce")
    # Условие: >80% значений — числа, и уникальных значений > MAX_UNIQUE
    # (иначе это скорее код/категория, а не количественный показатель)
    if numeric.notna().mean() > 0.8 and df[col].nunique() > MAX_UNIQUE:
        num_cols.append(col)

print(f"Числовых колонок: {len(num_cols)}")

if num_cols:
    # describe() — сводная статистика: count, mean, std, min, 25%, 50%, 75%, max
    df_num = df[num_cols].apply(pd.to_numeric, errors="coerce")
    display(df_num.describe().round(2))

    # Проверка на отрицательные значения
    for col in num_cols:
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        neg = (vals < 0).sum()
        if neg > 0:
            print(f"\n⚠ {col}: отрицательных значений {neg:,}")
else:
    print("Числовых колонок не обнаружено (или все попали в категориальные).")

## 4. Сопоставление с CRM

Загружаем CRM, извлекаем ИНН и оцениваем пересечение с ППЗ.

In [ ]:
# ── Сопоставление ИНН между ППЗ и CRM ──
# Цель: понять, насколько пересекаются компании в двух источниках.
# Если пересечение маленькое — данные ППЗ дополняют CRM новыми компаниями.
# Если большое — можно обогащать CRM-записи данными из ППЗ (напр. Desc_text).

if INN_COL is None:
    print("⚠ Колонка с ИНН не определена — сопоставление невозможно.")
else:
    # Загружаем из CRM только колонку X_INN (экономим память)
    crm = pd.read_csv(CRM_FILE, dtype=str, low_memory=False, usecols=["X_INN"])

    # Нормализуем ИНН одинаковой функцией, чтобы сравнение было корректным
    # (иначе "7701234567" и "07701234567" не совпадут)
    crm_inns = set(crm["X_INN"].apply(normalize_inn).dropna())
    ppz_inns = set(df[INN_COL].apply(normalize_inn).dropna())

    # Теоретико-множественные операции:
    overlap  = crm_inns & ppz_inns   # пересечение — компании в обоих источниках
    only_crm = crm_inns - ppz_inns   # только в CRM, но нет в ППЗ
    only_ppz = ppz_inns - crm_inns   # только в ППЗ, но нет в CRM

    summary = pd.DataFrame([
        ("CRM",  f"{len(crm_inns):,}",  f"{len(overlap):,}",  f"{len(only_crm):,}"),
        ("ППЗ",  f"{len(ppz_inns):,}",  f"{len(overlap):,}",  f"{len(only_ppz):,}"),
    ], columns=["Источник", "Уник. ИНН", "Пересечение", "Только здесь"])

    display(summary.style.hide(axis="index"))

    # Процент покрытия: какая доля ИНН из ППЗ нашлась в CRM (и наоборот)
    pct_ppz_in_crm = len(overlap) / max(len(ppz_inns), 1) * 100
    pct_crm_in_ppz = len(overlap) / max(len(crm_inns), 1) * 100
    print(f"\n{pct_ppz_in_crm:.1f}% ИНН из ППЗ присутствуют в CRM")
    print(f"{pct_crm_in_ppz:.1f}% ИНН из CRM присутствуют в ППЗ")

## 5. Оценка колонки для Desc_text

Идентифицируем колонку с результатом урегулирования ИПУ
и оцениваем её заполненность.

In [ ]:
# ── Поиск колонки с результатом урегулирования ИПУ (аналог Desc_text) ──
# В CRM поле Desc_text (COMMENT_TEXT) плохо заполнено.
# Реестр ППЗ должен содержать эту информацию, но мы не знаем точное
# название колонки. Ищем по ключевым словам в названии.

desc_candidates = [c for c in df.columns
                   if any(kw in c.lower()
                          for kw in ["desc", "описани", "результат", "урегул",
                                     "ипу", "коммент", "примечан", "решен",
                                     "текст", "причин"])]

print(f"Колонки-кандидаты для Desc_text: {desc_candidates}")

if desc_candidates:
    for col in desc_candidates:
        filled = df[col].notna().sum()
        pct = filled / len(df) * 100
        uniq = df[col].nunique()
        print(f"\n--- {col} ---")
        print(f"  Заполнено: {filled:,} из {len(df):,} ({pct:.1f}%)")
        print(f"  Уникальных значений: {uniq:,}")

        # Заполненность по годам — важно: если в 2023 заполнено 90%,
        # а в 2025 — 10%, значит качество данных деградирует
        by_year = df.groupby("source_year")[col].apply(
            lambda x: f"{x.notna().sum():,} / {len(x):,} ({x.notna().mean()*100:.1f}%)"
        )
        print(f"  По годам:")
        for y, v in by_year.items():
            print(f"    {y}: {v}")

        # Примеры значений — чтобы визуально оценить содержимое
        sample = df[col].dropna().head(5)
        if len(sample) > 0:
            print(f"  Примеры:")
            for i, val in enumerate(sample, 1):
                preview = str(val)[:120]
                print(f"    {i}. {preview}{'...' if len(str(val)) > 120 else ''}")
else:
    # Если автопоиск не нашёл — выводим все колонки с процентом заполненности,
    # чтобы пользователь мог вручную определить нужную
    print("\n⚠ Автоматически подходящая колонка не найдена.")
    print("Все колонки:")
    for i, col in enumerate(df.columns):
        filled = df[col].notna().sum()
        pct = filled / len(df) * 100
        print(f"  {i+1:2d}. {col:40s} — заполнено {pct:.0f}%")

### 5.1. Детальная проверка «Результат урегулирования по ИПУ»

Прямая проверка без автодетекта — берём колонку по точному имени
и смотрим на содержимое «вручную».

In [ ]:
# ── Прямая проверка колонки «Результат урегулирования по ИПУ» ──
# Без автодетекта — берём по точному имени или ищем подстроку.

TARGET_COL = None
for c in df.columns:
    if "урегулирования по ИПУ" in str(c):
        TARGET_COL = c
        break

if TARGET_COL is None:
    print("⚠ Колонка с подстрокой 'урегулирования по ИПУ' не найдена.")
    print("Все колонки:")
    for i, c in enumerate(df.columns):
        print(f"  {i}: {c}")
else:
    print(f"Колонка: '{TARGET_COL}'")
    total = len(df)
    raw = df[TARGET_COL]

    # 1. Базовая статистика заполненности
    is_null = raw.isna()
    is_empty_str = raw.fillna("").str.strip() == ""
    truly_empty = is_null | is_empty_str

    filled = (~truly_empty).sum()
    print(f"\nВсего строк: {total:,}")
    print(f"NaN (пропуски):          {is_null.sum():,}")
    print(f"Пустые строки ('', ' '): {(is_empty_str & ~is_null).sum():,}")
    print(f"Итого пусто:             {truly_empty.sum():,} ({truly_empty.mean()*100:.1f}%)")
    print(f"Заполнено:               {filled:,} ({filled/total*100:.1f}%)")

    # 2. Заполненность по годам
    print(f"\n--- По годам ---")
    for year in sorted(df["source_year"].unique()):
        mask = df["source_year"] == year
        yr_total = mask.sum()
        yr_filled = (~truly_empty[mask]).sum()
        print(f"  {year}: {yr_filled:,} / {yr_total:,} ({yr_filled/yr_total*100:.1f}%)")

    # 3. Уникальные непустые значения
    non_empty = raw[~truly_empty]
    print(f"\n--- Уникальных непустых значений: {non_empty.nunique():,} ---")

    # 4. Все значения с частотой (если немного) или топ-30
    vc = non_empty.value_counts()
    if len(vc) <= 50:
        print(f"\nВСЕ значения:")
        for val, cnt in vc.items():
            preview = str(val)[:150]
            print(f"  [{cnt:,}x] {preview}")
    else:
        print(f"\nТоп-30 значений (из {len(vc):,}):")
        for val, cnt in vc.head(30).items():
            preview = str(val)[:150]
            print(f"  [{cnt:,}x] {preview}")

    # 5. Примеры строк, где колонка заполнена (первые 10)
    if filled > 0:
        print(f"\n--- Примеры заполненных строк (первые 10) ---")
        sample_rows = df[~truly_empty].head(10)
        for i, (idx, row) in enumerate(sample_rows.iterrows(), 1):
            inn_val = row.get(INN_COL, "?") if INN_COL else "?"
            val = str(row[TARGET_COL])[:200]
            print(f"  {i}. ИНН={inn_val} | {val}")

## 6. Сводка проверок

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ИТОГОВАЯ СВОДКА ВСЕХ ПРОВЕРОК
# ══════════════════════════════════════════════════════════════════
# Собираем результаты всех предыдущих проверок в одну таблицу
# с цветовой индикацией:
#   OK      (зелёный)  — проверка пройдена
#   WARNING (жёлтый)   — есть замечания, требуется внимание
#   ERROR   (красный)  — критическая проблема, нужно исправить

checks = []

# 1. Совпадают ли колонки во всех 3 файлах?
col_match = all(col_sets[y] == col_sets[years[0]] for y in years)
checks.append(("Совпадение колонок по годам",
                "OK" if col_match else "WARNING",
                "Колонки идентичны" if col_match else "Есть расхождения"))

# 2. Все ли даты в допустимом периоде 2023–2025?
date_ok = True
date_note = ""
for col in date_cols:
    dt = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
    valid = dt.dropna()
    out = ((valid < DATE_FROM) | (valid > DATE_TO)).sum()
    if out > 0:
        date_ok = False
        date_note += f"{col}: {out:,} за пределами; "
if not date_cols:
    checks.append(("Период данных 2023–2025", "WARNING", "Даты не обнаружены"))
else:
    checks.append(("Период данных 2023–2025",
                    "OK" if date_ok else "WARNING",
                    "В диапазоне" if date_ok else date_note.strip()))

# 3. Есть ли колонки, заполненные менее чем на 50%?
max_null_pct = null_report["% пропусков"].max()
checks.append(("Пропуски",
                "OK" if max_null_pct < 50 else "WARNING",
                f"Макс. пропусков: {max_null_pct:.1f}%"))

# 4. Есть ли полные дубликаты строк?
checks.append(("Полные дубликаты",
                "OK" if full_dupes == 0 else "WARNING",
                f"{full_dupes:,}" if full_dupes > 0 else "Нет"))

# 5. Корректность ИНН (пропуски, длина, символы)
if INN_COL:
    inn_ok = null_inn == 0 and len(bad_len) == 0 and len(non_digit) == 0
    inn_note = "Без ошибок" if inn_ok else f"null={null_inn}, некорр.длина={len(bad_len)}, нецифр.={len(non_digit)}"
    checks.append(("Формат ИНН", "OK" if inn_ok else "ERROR", inn_note))
else:
    checks.append(("Формат ИНН", "WARNING", "Колонка не найдена"))

# 6. Какая доля ИНН из ППЗ пересекается с CRM?
if INN_COL:
    checks.append(("Пересечение с CRM",
                    "OK" if pct_ppz_in_crm > 50 else "WARNING",
                    f"{pct_ppz_in_crm:.1f}% ИНН ППЗ найдены в CRM"))

# Формируем таблицу и раскрашиваем статусы
summary_df = pd.DataFrame(checks, columns=["Проверка", "Статус", "Комментарий"])

def color_status(val):
    """Цветовая индикация: зелёный/жёлтый/красный."""
    if val == "OK":
        return "background-color: #d4edda; color: #155724"
    elif val == "WARNING":
        return "background-color: #fff3cd; color: #856404"
    elif val == "ERROR":
        return "background-color: #f8d7da; color: #721c24"
    return ""

display(
    summary_df.style
    .hide(axis="index")
    .map(color_status, subset=["Статус"])
)

print(f"\nИтого: {len(df):,} строк объединено из {len(frames)} файлов.")
print(f"Результат сохранён в {OUT_FILE}")